### Prepare Retrain Data

Selects low-confidence crops from a completed inference run and copies them for human labeling.
This is a deliberate manual step — nothing is labeled automatically.

**Workflow:** `infer_cropbased` → **this notebook** → label with `crop_labeler.py` / `relabel.py` → move to `annotated_crops/` → `retrain_cropbased`

**Input** — `outputs/inference/crop_results/{RUN_NAME}/results.csv` + crop images  
**Output** — `outputs/training/retrain_review/{class}/` (crops to review)

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `INFER_RESULTS` | Empty output if path wrong — set to a specific run folder or leave as `CROP_RESULTS_ROOT` to scan all runs |

**Optional (Cell 2):** `CONF_THRESHOLD_LOW` (0.70 — below this = uncertain, queue for review), `CONF_THRESHOLD_HIGH` (0.95 — above this = skip), `MAX_PER_CLASS` (200), `INCLUDE_BACKGROUND` (True), `FORCE_ALL` (False — set True to queue every crop regardless of confidence)

> **Skip this notebook** if the run produced fewer than ~300 crops — go straight to `relabel.py`.


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← no edits needed
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR         = BASE_DIR / 'models'
LABELED_DIR       = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W       = BASE_DIR / 'InsectNet' / 'model.pth'

# Inference results: read from local SSD on Colab, or from BASE_DIR locally.
LOCAL_INF  = Path('/content/data') if IN_COLAB else BASE_DIR
CROP_RESULTS_ROOT = LOCAL_INF / 'outputs' / 'inference' / 'crop_results'
YOLO_RESULTS_ROOT = LOCAL_INF / 'outputs' / 'inference' / 'yolo_results'

# Review output goes to local during Colab, then copy to Drive manually or via the final cell.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
REVIEW_DIR     = LOCAL_TRAINING / 'retrain_review'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR         : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'CROP_RESULTS_ROOT: {CROP_RESULTS_ROOT}  exists={CROP_RESULTS_ROOT.exists()}')
print(f'YOLO_RESULTS_ROOT: {YOLO_RESULTS_ROOT}  exists={YOLO_RESULTS_ROOT.exists()}')
print(f'LABELED_DIR      : {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'REVIEW_DIR       : {REVIEW_DIR}')


##### Cell 2 — Filter config  ← **edit this**

Controls which crops are selected for review:

- **`INFER_RESULTS`** — which inference run to draw from
- **`CONF_THRESHOLD_LOW`** — crops with confidence below this are flagged as uncertain
  and sent for review. Lower = more crops to review.
- **`INCLUDE_BACKGROUND`** — include background crops (useful as hard negatives for retraining)
- **`FORCE_ALL`** — send everything regardless of confidence (use sparingly)
- **`MAX_PER_CLASS`** — cap per class to keep the review session manageable

In [ ]:
# ── Which pipeline results to use ───────────────────────────────
# Point to the results folder from infer_cropbased
# e.g. RESULTS_ROOT / 'results_five_class' or just RESULTS_ROOT
INFER_RESULTS = CROP_RESULTS_ROOT

# ── Confidence thresholds ────────────────────────────────────────
# Crops with confidence BELOW this are 'unsure' -> good candidates for review
CONF_THRESHOLD_LOW  = 0.70   # below this = unsure, send for review
CONF_THRESHOLD_HIGH = 0.95   # above this = confident, skip review (unless forced)

# ── Which classes to include ─────────────────────────────────────
INCLUDE_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'background']

# ── Include background crops? ────────────────────────────────────
# Background crops from background/ folder are useful as hard negatives
INCLUDE_BACKGROUND = True

# ── Force include all (ignore confidence filter) ─────────────────
FORCE_ALL = False

# ── Max crops per class (to keep review manageable) ──────────────
MAX_PER_CLASS = 200   # None = no limit

print('Filter config ready.')

# ── Where labeled crops will go ────────────────────────────────────────
# Name of the sub-folder inside data/training/annotated_crops/ to write to.
# Change this to match your labeler id or session, e.g. 'labeled_mb'.
LABEL_DEST_NAME = 'labeled_ls'  # ← change if needed
LABEL_DEST = LABELED_DIR / LABEL_DEST_NAME


##### Cell 3 — Load and inspect results

Reads all `results.csv` files from the selected run.
Shows confidence distribution per class so you can tune `CONF_THRESHOLD_LOW`
before deciding what to send for review.

In [ ]:
import csv
import numpy as np
from collections import defaultdict

all_rows = []
for csv_path in sorted(INFER_RESULTS.rglob('results.csv')):
    with open(csv_path, newline='') as f:
        for row in csv.DictReader(f):
            if row.get('pollinator_detected') not in ('yes', 'no'): continue
            row['_csv_path'] = str(csv_path)
            row['_camera']   = csv_path.parent.name
            all_rows.append(row)

print(f'Total rows loaded: {len(all_rows)}')

# ── Confidence distribution ──────────────────────────────────────
detected = [r for r in all_rows if r.get('pollinator_detected') == 'yes']
print(f'\nDetected (insect): {len(detected)}')

by_class = defaultdict(list)
# Support both modern prefixed columns (any __pollinator_type) and legacy name
def _get_pollinator_type(row, default='background'):
    for k, v in row.items():
        if k.endswith('__pollinator_type') and v:
            return v
    return row.get('pollinator_type', '') or default

for r in detected:
    pt = _get_pollinator_type(r, default='background')
    by_class[pt].append(r)

print('\nClass breakdown + confidence stats:')
print(f'  {"Class":15}  {"Count":>6}  {"AvgConf":>8}  {"<threshold":>10}')
for cls, rows in sorted(by_class.items()):
    confs = []
    for r in rows:
        try:
            # Try modern prefixed columns first, then legacy names
            _cv = None
            for _k, _v in r.items():
                if _k.endswith('__group_conf') and _v:
                    _cv = _v; break
            if _cv is None:
                for _k, _v in r.items():
                    if _k.endswith('__binary_conf') and _v:
                        _cv = _v; break
            if _cv is None:
                _cv = r.get('group_confidence') or r.get('binary_confidence') or 0
            confs.append(float(_cv))
        except: pass
    if confs:
        avg  = np.mean(confs)
        low  = sum(1 for c in confs if c < CONF_THRESHOLD_LOW)
        print(f'  {cls:15}  {len(rows):>6}  {avg:>8.3f}  {low:>10}')

# Background crops
bg_crops = list(INFER_RESULTS.rglob('background/*.jpg'))
print(f'\nBackground crops: {len(bg_crops)}')


##### Cell 4 — Preview

Shows exactly how many crops will be sent per class **before** copying anything.
Review this table and adjust the filter config in Cell 2 if needed.
**Nothing is copied yet** — this is just a preview.

In [ ]:
to_review = defaultdict(list)  # class -> list of (crop_path, row)

# ── Insect crops (from results.csv) ─────────────────────────────
for r in detected:
    pt   = _get_pollinator_type(r, default='other')
    if pt not in INCLUDE_CLASSES: continue
    fname = r.get('crop_filename', '')
    if not fname: continue
    camera = r['_camera']
    crop_p = INFER_RESULTS / camera / 'crops' / fname
    if not crop_p.exists(): continue

    try:
        # Try modern prefixed columns first; fall back to legacy names
        conf_val = None
        for _k, _v in r.items():
            if _k.endswith('__group_conf') and _v:
                conf_val = _v; break
        if conf_val is None:
            for _k, _v in r.items():
                if _k.endswith('__binary_conf') and _v:
                    conf_val = _v; break
        if conf_val is None:
            conf_val = r.get('group_confidence') or r.get('binary_confidence') or 0
        conf = float(conf_val)
    except: conf = 0.0

    if FORCE_ALL or conf < CONF_THRESHOLD_LOW:
        to_review[pt].append((crop_p, conf, r))

# ── Background crops ─────────────────────────────────────────────
if INCLUDE_BACKGROUND:
    for bg_p in bg_crops:
        to_review['background'].append((bg_p, 0.0, {}))

# ── Apply max per class ──────────────────────────────────────────
# Sort by confidence ascending (most uncertain first)
print('Preview — crops to send for review:')
print(f'  {"Class":15}  {"Available":>10}  {"Will send":>10}')
total_send = 0
for cls in sorted(to_review):
    items = sorted(to_review[cls], key=lambda x: x[1])  # lowest conf first
    if MAX_PER_CLASS: items = items[:MAX_PER_CLASS]
    to_review[cls] = items
    total_send += len(items)
    print(f'  {cls:15}  {len(to_review[cls]):>10}  {len(items):>10}')
print(f'\nTotal: {total_send} crops')


##### Cell 5 — Copy to review folder

Shows the final plan. **Still nothing copied yet.**
Run **Cell 6** (the Execute cell) to actually copy the files.

In [ ]:
import shutil

# Confirm before copying
print(f'About to copy {total_send} crops to:')
print(f'  {REVIEW_DIR}')
print()
print('Structure will be:')
for cls in sorted(to_review):
    print(f'  retrain_review/{cls}/  ({len(to_review[cls])} crops)')
print()
print('Run Cell 6 to confirm and execute.')


##### Cell 6 — Execute copy  ← **run this to actually copy crops**

Copies the selected crops to `outputs/training/retrain_review/`. Nothing is moved before this cell runs.

In [ ]:
# ── EXECUTE — run this cell to actually copy ─────────────────────
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

copied = defaultdict(int)
skipped = 0

for cls, items in to_review.items():
    dest_dir = REVIEW_DIR / cls
    dest_dir.mkdir(exist_ok=True)
    for crop_p, conf, row in items:
        dest = dest_dir / crop_p.name
        if dest.exists():  # don't overwrite
            skipped += 1; continue
        shutil.copy2(str(crop_p), str(dest))
        copied[cls] += 1

print('Done. Copied:')
for cls, n in sorted(copied.items()):
    print(f'  {cls:15}: {n}')
if skipped: print(f'Skipped (already exist): {skipped}')
rel_review  = REVIEW_DIR.relative_to(BASE_DIR)
rel_labeled = LABEL_DEST.relative_to(BASE_DIR)
rel_labeled = LABEL_DEST.relative_to(BASE_DIR)
print(f'\nReview folder: {rel_review}')
print(f'\nNext: run crop_labeler.py on the review folder:')
print(f'  python3 tools/labeling/crop_labeler.py --results {rel_review}')
print(f'\nAfter labeling, move confirmed crops to:')
print(f'  {rel_labeled}')


##### Cell 7 — Check labeled_crops (optional)

Shows current counts in `data/training/annotated_crops/` per class.
Useful to see how much data you have before deciding whether to retrain.

In [ ]:
print('Current labeled_crops counts:')
total = 0
for cls_dir in sorted(LABELED_DIR.iterdir()):
    if not cls_dir.is_dir(): continue
    n = len(list(cls_dir.glob('*.jpg'))) + len(list(cls_dir.glob('*.jpeg')))
    print(f'  {cls_dir.name:15}: {n:>5}')
    total += n
print(f'  {"TOTAL":15}: {total:>5}')
